<a href="https://colab.research.google.com/github/Yogen4/Workshop.ai/blob/main/Week6(workshop6)Al_Ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from sklearn.metrics import classification_report

# Parameters
img_height = 224
img_width = 224
batch_size = 32

train_dir = "/content/drive/MyDrive/FruitinAmazon/train"
test_dir  = "/content/drive/MyDrive/FruitinAmazon/test"

# Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Load data
train = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical'
)

test = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

num_classes = len(train.class_indices)

# Model (Deeper + BN + Dropout)
model = models.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train
model.fit(train, validation_data=test, epochs=10)

# Evaluate
loss, acc = model.evaluate(test)
print("CNN Accuracy:", acc)

# Prediction + Report
pred = model.predict(test)
y_pred = np.argmax(pred, axis=1)

print(classification_report(test.classes, y_pred))

Found 90 images belonging to 6 classes.
Found 30 images belonging to 6 classes.
Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 27s 9s/step - accuracy: 0.2778 - loss: 18.9369 - val_accuracy: 0.2333 - val_loss: 1.8784
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 35s 7s/step - accuracy: 0.5000 - loss: 16.4627 - val_accuracy: 0.2667 - val_loss: 1.8380
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 18s 6s/step - accuracy: 0.5000 - loss: 13.1613 - val_accuracy: 0.1667 - val_loss: 3.4948
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 19s 6s/step - accuracy: 0.5889 - loss: 12.9442 - val_accuracy: 0.3000 - val_loss: 6.9407
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 20s 6s/step - accuracy: 0.6778 - loss: 7.8535 - val_accuracy: 0.2333 - val_loss: 13.4904
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 19s 6s/step - accuracy: 0.7333 - loss: 7.0692 - val_accuracy: 0.2000 - val_loss: 20.6076
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 18s 7s/step - accuracy: 0.6556 - loss: 8.6553 - val_accuracy: 0.2333 - val_loss: 26.3608
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 18s 6s/st

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers
from tensorflow.keras.models import Model
import numpy as np
from sklearn.metrics import classification_report

# Load VGG16
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

# Custom layers
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=x)

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
model.fit(train, validation_data=test, epochs=10)

# Evaluate
loss, acc = model.evaluate(test)
print("VGG16 Accuracy:", acc)

# Prediction + Report
pred = model.predict(test)
y_pred = np.argmax(pred, axis=1)

print(classification_report(test.classes, y_pred))

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 78s 28s/step - accuracy: 0.1667 - loss: 1.9476 - val_accuracy: 0.1667 - val_loss: 1.9349
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 77s 28s/step - accuracy: 0.1667 - loss: 1.9009 - val_accuracy: 0.1667 - val_loss: 1.9011
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 81s 28s/step - accuracy: 0.1667 - loss: 1.8603 - val_accuracy: 0.1667 - val_loss: 1.8744
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 74s 29s/step - accuracy: 0.1778 - loss: 1.8488 - val_accuracy: 0.1333 - val_loss: 1.8531
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 76s 27s/step - accuracy: 0.1444 - loss: 1.8268 - val_accuracy: 0.1000 - val_loss: 1.8351
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 76s 30s/step - accuracy: 0.1556 - loss: 1.7939 - val_accuracy: 0.1000 - val_loss: 1.8203
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 74s 29s/step - accuracy: 0.1778 - loss: 1.7750 - val_accuracy: 0.1333 - val_loss: 1.8083
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 76s 30s/step - accuracy: 0.2000 - loss: 1.7642 - val_accuracy: 0.2000 - val_loss: 1.7979
